In [1]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage,HumanMessage,AIMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

/tmp/ipykernel_14608/754196966.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder

In [3]:
llm=ChatGroq(model='qwen/qwen3.8-27b',temperature=0.2)

In [4]:
embeddings=OpenAIEmbeddings(model='text-embedding-3-small')

In [8]:
docs=WebBaseLoader('https://www.flipkart.com/apple-iphone-17-pro-max-deep-blue-256-gb/p/itm81f3a173391c4').load()

In [9]:
spiltter_obj=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=100)

In [10]:
chunks=spiltter_obj.split_documents(docs)

In [11]:
vector_store=Chroma(embedding_function=embeddings,persist_directory="./chroma",collection_name='mobile_info_new')

In [12]:
vector_store.add_documents(chunks)

['d319e736-c6bd-44f3-8f1d-7be7a7d46145',
 'b4ee116a-169e-4df7-8939-93a51478fbee',
 '6df1a77d-d382-423c-a70c-3c1bfd67d0b9',
 'd983800f-6d02-4412-85d6-ece99638f9f0',
 '1e44039c-bc7a-4658-ba03-66a388195ad6',
 '3a390362-14c7-459d-9b33-4cb71df416bf',
 'bc181a4b-903d-4f6a-91fc-edfd06e29fb5',
 '9cb3d5f9-965e-417c-9964-ccb453e812fd',
 '09dd7ba6-f7a1-4f88-a39b-6c9892cac244',
 '753c4f80-b9da-4fa6-ab4f-fc024d2b2621',
 '272fe9f2-603a-426b-988d-1f5e3f732532',
 '67aaea61-98c0-44e5-a1da-cf2e09cac70e']

In [13]:
retriever=vector_store.as_retriever()

In [17]:
system_prompt='''You are a helpful Q/A system,Answer the Query from the provided context-{context}.
If you dont know the answer say I dont know as answer and Thank you!.keep the answer concise. '''

In [18]:
prompt=ChatPromptTemplate.from_messages([
    ('system',system_prompt),
    ('human','{input}')
])

In [23]:
context_qa_prompt='Given a chat history and the latest user question which might reference context in chat history and dont answer without looking into chat history'

In [39]:
context_prompt=ChatPromptTemplate.from_messages([
    ('system','Reprase the query as standlone query based on past context and chat history'),
    MessagesPlaceholder('chat_history'),
    ('human','{input}')

])

In [ ]:
qa_prompt=ChatPromptTemplate.from_messages([
    ('system',context_qa_prompt),
    MessagesPlaceholder('chat_history'),
    ('human','{input} \n\n context: \n {context}')

])

In [41]:
history_awr_retriever=create_history_aware_retriever(llm,retriever,context_prompt)

In [42]:
qa_stuff_chain=create_stuff_documents_chain(llm,qa_prompt)

In [43]:
rag_chain=create_retrieval_chain(history_awr_retriever,qa_stuff_chain)

In [59]:
chat_history=[]

In [79]:
question="whats the last second question i have asked"

In [80]:
result=rag_chain.invoke({'input':question,'chat_history':chat_history})
result

{'input': 'whats the last second question i have asked',
 'chat_history': [HumanMessage(content='whats the context about.Give me 5 words answer', additional_kwargs={}, response_metadata={}),
  AIMessage(content='iPhone 17 Pro Max specs', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='whats the price?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='₹1,49,900', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='is there any offers or discounts  for Iphone !& pro max?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Yes, ₹7,000 bank offer available.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='only one?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='No, exchange and coupons too.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_c

In [78]:
chat_history.extend(
    [HumanMessage(content=question),
     AIMessage(content=result['answer'])
     ]
)

In [75]:
chat_history

[HumanMessage(content='whats the context about.Give me 5 words answer', additional_kwargs={}, response_metadata={}),
 AIMessage(content='iPhone 17 Pro Max specs', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats the price?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='₹1,49,900', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='is there any offers or discounts  for Iphone !& pro max?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Yes, ₹7,000 bank offer available.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='only one?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='No, exchange and coupons too.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]